# AI-Based Network Intrusion Detection System
## Notebook 01: Exploratory Data Analysis (EDA)
**Dataset**: UNSW-NB15 Benchmark (Australian Centre for Cyber Security)
**Objective**: Comprehensive statistical analysis, class distribution profiling, and multicollinearity investigation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Configure aesthetic styles
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

### 1. Load Cleaned Dataset Splits (Preserving Benchmark Partition)

In [ ]:
train_df = pd.read_csv("../data/processed/train_cleaned.csv")
test_df = pd.read_csv("../data/processed/test_cleaned.csv")
print(f"Training set shape: {train_df.shape}")
print(f"Testing set shape:  {test_df.shape}")
print(f"Total records:      {len(train_df) + len(test_df):,}")

### 2. Binary Class Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
train_df["label"].value_counts().plot(kind="bar", ax=axes[0], color=["#d95f02", "#2b5c8f"], edgecolor="black")
axes[0].set_title("Training Set: Binary Class Counts")
axes[0].set_xticklabels(["Attack (1)", "Normal (0)"], rotation=0)
axes[0].set_ylabel("Count")

test_df["label"].value_counts().plot(kind="bar", ax=axes[1], color=["#d95f02", "#2b5c8f"], edgecolor="black")
axes[1].set_title("Testing Set: Binary Class Counts")
axes[1].set_xticklabels(["Attack (1)", "Normal (0)"], rotation=0)
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()

### 3. Multiclass Attack Categories Breakdown

In [ ]:
plt.figure(figsize=(10, 6))
train_df["attack_cat"].value_counts().plot(kind="barh", color="#2b5c8f", edgecolor="black")
plt.title("Attack Category Breakdown in Training Partition")
plt.xlabel("Count")
plt.gca().invert_yaxis()
plt.show()

### 4. Correlation Analysis & Feature Relationship Study
**Key Insight**: Rather than blindly discarding collinear features, tree models (Random Forest, XGBoost) leverage individual splits orthogonally, whereas L2 regularization prevents instability in Logistic Regression.

In [ ]:
num_cols = train_df.select_dtypes(include=[np.number]).columns.drop("label")
corr = train_df[num_cols].corr()

high_corr = []
for i in range(len(num_cols)):
    for j in range(i + 1, len(num_cols)):
        r = corr.loc[num_cols[i], num_cols[j]]
        if abs(r) >= 0.90:
            high_corr.append((num_cols[i], num_cols[j], r))

corr_df = pd.DataFrame(high_corr, columns=["Feature 1", "Feature 2", "Pearson r"]).sort_values(by="Pearson r", ascending=False, key=abs)
print(f"Identified {len(corr_df)} pairs with |r| >= 0.90:")
display(corr_df.head(10))